# Lighting, Color & Sampling Demo

This notebook walks through the lowest-level building blocks: smooth random fields, bilinear sampling, palettes, lighting, and finishing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math_vis import smooth_field, sample, squash, make_cmap, cosine_cmap, upfield, finish, apply_glass_lighting, apply_lead_lines

SEED = 3
W, H = 300, 300
rng = np.random.default_rng(SEED)

In [ ]:
# 1. Build a small random field and a full-size version
small = rng.normal(size=(40, 40))
field = smooth_field((40, 40), 8, rng)
big = upfield(field, W, H)

fig, axs = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, title in zip(axs, [small, field, big], ['raw noise', 'smooth_field', 'upfield to full size']):
    im = ax.imshow(data, cmap='gray', origin='lower')
    ax.set_title(title); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

In [ ]:
# 2. Sample the field at fractional positions
rows = np.linspace(0, 39, H)
cols = np.linspace(0, 39, W)
RR, CC = np.meshgrid(rows, cols, indexing='ij')
resampled = sample(field, RR, CC)

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].imshow(big, cmap='gray', origin='lower'); axs[0].set_title('PIL resize'); axs[0].axis('off')
axs[1].imshow(resampled, cmap='gray', origin='lower'); axs[1].set_title('map_coordinates sample'); axs[1].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 3. squash maps unbounded values to [0,1]
x = np.linspace(-4, 4, 500)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, squash(x), label='softness=1.5')
ax.plot(x, squash(x, softness=3.0), label='softness=3.0')
ax.axhline(0.5, color='gray', ls='--')
ax.set_title('squash: tanh-based map to [0,1]'); ax.legend(); plt.show()

In [ ]:
# 4. Palettes
hex_cmap = make_cmap(['#0b1d3a', '#1f4e8c', '#58d3c0', '#f4e285', '#e0453a'])
cos_cmap = cosine_cmap(a=(0.6, 0.55, 0.62), b=(0.38, 0.38, 0.34), c=(0.85, 0.85, 0.85), d=(0.0, 0.30, 0.62))

fig, axs = plt.subplots(2, 1, figsize=(10, 4))
t = np.linspace(0, 1, 256)
axs[0].imshow(hex_cmap(t)[None, :, :3], aspect='auto'); axs[0].set_title('make_cmap from hex list'); axs[0].set_yticks([])
axs[1].imshow(cos_cmap(t)[None, :, :3], aspect='auto'); axs[1].set_title('cosine_cmap: a + b*cos(2π(c*t+d))'); axs[1].set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
# 5. Map a smooth field to colors
color_field = smooth_field((60, 60), 10, rng)
tmap = squash(upfield(color_field, W, H))
img_hex = hex_cmap(tmap)[:, :, :3] * 255
img_cos = cos_cmap(tmap)[:, :, :3] * 255

fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(img_hex.astype(np.uint8)); axs[0].set_title('hex palette'); axs[0].axis('off')
axs[1].imshow(img_cos.astype(np.uint8)); axs[1].set_title('cosine palette'); axs[1].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 6. Lighting on a few synthetic cells
from scipy.spatial import cKDTree
n = 12
seeds = rng.uniform([0, 0], [W, H], size=(n, 2))
X, Y = np.meshgrid(np.arange(W)+0.5, np.arange(H)+0.5)
P = np.column_stack([X.ravel(), Y.ravel()])
d, idx = cKDTree(seeds).query(P, k=2)
cell = idx[:, 0]
t_cell = rng.uniform(0, 1, n)
flat = hex_cmap(t_cell[cell])[:, :3] * 255
lit = apply_glass_lighting(flat, P, seeds, d[:, 0], cell, facet=0.6, glow=0.5)
leaded = apply_lead_lines(lit, d[:, 0], d[:, 1], W, H, lead_px=3, rng=rng)

fig, axs = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, img, title in zip(axs, [flat, lit, leaded], ['flat color', '+ facet/glow lighting', '+ lead lines']):
    ax.imshow(np.clip(img, 0, 255).reshape(H, W, 3).astype(np.uint8))
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 7. Finishing effects: grain and vignette
polished = finish(leaded.reshape(H, W, 3), grain=3, vignette=0.4, display=False)
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(np.clip(leaded,0,255).reshape(H,W,3).astype(np.uint8)); axs[0].set_title('before finish'); axs[0].axis('off')
axs[1].imshow(polished); axs[1].set_title('after finish (grain + vignette)'); axs[1].axis('off')
plt.tight_layout(); plt.show()